In [6]:
import traceback
import os, shutil, tempfile
os.environ["HYDRA_FULL_ERROR"]="1"

import warnings
from collections.abc import Sequence
import subprocess

# def performance_tweaks():
#     import torch
#     torch.backends.cudnn.benchmark = True
#     torch.backends.cudnn.deterministic = False
#     torch.use_deterministic_algorithms(False)
#     torch.set_float32_matmul_precision('high')
#     torch.jit.enable_onednn_fusion(True)
#     torch.autograd.set_detect_anomaly(False, False) # type:ignore
#     torch.autograd.profiler.profile(False) # type:ignore
#     torch.autograd.profiler.emit_nvtx(False) # type:ignore

# performance_tweaks() # this has 0 effect btw

# PYTHON_BIN = "/home/jj/miniconda3/envs/mapperatorinator/bin/python"
MAPINATOR_DIR = "/run/media/jj/Ext2TB/Files/Extra/Programming2/Libs/Mapperatorinator"
PYTHON_BIN = f"{MAPINATOR_DIR}/Mapperatorinator/.venv/bin/python"
SUPER_TIMING = True
# ---------------------------------------------------------------------------- #
#                                  definitions                                 #
# ---------------------------------------------------------------------------- #

# [
#     "/home/jj/miniconda3/envs/mapperatorinator/bin/python",
#     "inference.py",
#     "-cn",
#     "v30",
#     "audio_path='/var/mnt/ssd/Files/Documents/Музыка/a.mp3'",
#     "output_path='/var/mnt/ssd/Files/Programming/libs/mapperatorinator'",
#     "gamemode=0",
#     "difficulty=5",
#     "year=2023",
#     "hp_drain_rate=5",
#     "circle_size=4",
#     "overall_difficulty=8",
#     "approach_rate=9",
#     "slider_multiplier=1.4",
#     "slider_tick_rate=1",
#     "keycount=4",
#     "cfg_scale=1.0",
#     "temperature=0.9",
#     "top_p=0.9",
#     "export_osz=true",
#     "add_to_beatmap=false",
#     "hitsounded=true",
#     "super_timing=true",
# ]

def _to_string_seq(x):
    if x is None: return ()
    if isinstance(x, str): return (x, )
    return x

def get_command(
    name: str,
    input:str,
    output:str,
    difficulty: float = 5,
    mapper_id: int | str | None = None,
    descriptors: str | Sequence[str] | None = (), # ["jump aim", "clean"]
    negative_descriptors: str | Sequence[str] | None = (), # from discord ["improvisation","tech","bursts"]
    cfg_scale: float | None = None, # 1 # Scale of classifier-free guidance
    temperature: float | None = None, # 0.9
    top_p: float | None = None, # 0.9
    year: int | None = 2024, # Always provide a year argument between 2007 and 2024. If you leave it unknown, the model might generate with an inconsistent style.

    super_timing: bool | None = None,

    approach_rate: float | None = None, # 9
    circle_size: float | None = None, # 4
    hp_drain_rate: float | None = None, # 5
    overall_difficulty: float | None = None, # 8
    slider_multiplier: float | None = None, # 1.4,
    slider_tick_rate: float | None = None, # 1

    timer_bpm_threshold: float | None = None, # 0.7
    generate_positions: bool | None = None, # false #  Use diffusion to generate object positions
    refine_iters: int | None = None, # 10, Number of diffusion refinement iterations

    lora_path: str | None = None,
):
    descriptors = _to_string_seq(descriptors)
    negative_descriptors = _to_string_seq(negative_descriptors)

    commands = [
        PYTHON_BIN,
        "inference.py",
        "-cn", "v32", # load v32 config
        "device='cuda'",
        # "compile=false",
        f"audio_path='{input}'",
        f"output_path='{output}'",
        "gamemode=0",
        f"difficulty={difficulty}",
        "export_osz=true",
        "add_to_beatmap=false",
        "hitsounded=true",
    ]

    sq = "'"
    if len(descriptors) > 0:
        s = f"'{f'{sq},{sq}'.join(descriptors)}'"
        commands.append(f'descriptors=[{s}]')

    if len(negative_descriptors) > 0:
        s = f"'{f'{sq},{sq}'.join(negative_descriptors)}'"
        commands.append(f'negative_descriptors=[{s}]')

    if lora_path is not None: commands.insert(6, f"lora_path='{lora_path}'")
    if year is not None: commands.append(f"year={year}")
    if mapper_id is not None: commands.append(f"mapper_id={mapper_id}")
    if super_timing is not None: commands.append(f"super_timing={str(bool(super_timing)).lower()}")

    if cfg_scale is not None: commands.append(f"cfg_scale={cfg_scale}")
    if temperature is not None: commands.append(f"temperature={temperature}")
    if top_p is not None: commands.append(f"top_p={top_p}")

    if approach_rate is not None: commands.append(f"approach_rate={approach_rate}")
    if circle_size is not None: commands.append(f"circle_size={circle_size}")
    if hp_drain_rate is not None: commands.append(f"hp_drain_rate={hp_drain_rate}")
    if overall_difficulty is not None: commands.append(f"overall_difficulty={overall_difficulty}")
    if slider_multiplier is not None: commands.append(f"slider_multiplier={slider_multiplier}")
    if slider_tick_rate is not None: commands.append(f"slider_tick_rate={slider_tick_rate}")
    if timer_bpm_threshold is not None: commands.append(f"timer_bpm_threshold={timer_bpm_threshold}")
    if generate_positions is not None: commands.append(f"generate_positions={str(bool(generate_positions)).lower()}")
    if refine_iters is not None: commands.append(f"refine_iters={refine_iters}")

    # determine artist title
    filename = '.'.join(os.path.basename(input).split('.')[:-1])
    if ' - ' in filename:
        artist, title = filename.split(' - ')
    else:
        artist = ''
        title = filename

    commands.append(f"artist='{artist}'")
    commands.append(f"title='{title}'")
    commands.append(f"creator='Mapperatorinator ({name})'")

    print(" ".join(commands))
    return commands


# ---------------------------------------------------------------------------- #
#                                    presets                                   #
# ---------------------------------------------------------------------------- #
presets = []

# ---------------------------------- default-2023 --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name='2023 6.0',
    year=2023,
    super_timing=SUPER_TIMING,
    approach_rate=10,
    difficulty=6,
))

# ---------------------------------- default --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name='default 6.0',
    year=2024,
    super_timing=SUPER_TIMING,
    approach_rate=10,
    difficulty=6,
))

# ---------------------------------- jumps2 --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    descriptors=['skillset/jumps', 'style/clean'],
    name='clean jumps 6.0 cfg2',
    year=2024,
    super_timing=SUPER_TIMING,
    approach_rate=10,
    difficulty=6,
    cfg_scale=2,
))


# ---------------------------------- jumps4 --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    descriptors=['skillset/jumps', 'style/clean'],
    name='clean jumps 6.0 cfg4',
    year=2024,
    super_timing=SUPER_TIMING,
    approach_rate=10,
    difficulty=6,
    cfg_scale=4,
))

# ---------------------------------- Kroytz --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name='LoRA-Kroytz 6.0',
    year=2024,
    super_timing=SUPER_TIMING,
    approach_rate=10,
    difficulty=6,
    lora_path='OliBomby/Mapperatorinator-v32-LoRA-Kroytz'
))

# ---------------------------------- Kroytz-jumps --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    descriptors=['skillset/jumps', 'style/clean'],
    name='LoRA-Kroytz clean jumps 6.0 cfg2',
    year=2024,
    super_timing=SUPER_TIMING,
    approach_rate=10,
    difficulty=6,
    cfg_scale=2,
    lora_path='OliBomby/Mapperatorinator-v32-LoRA-Kroytz'
))


# ---------------------------------------------------------------------------- #
#                                postprocessing                                #
# ---------------------------------------------------------------------------- #
def packgod(folder, out_dir):
    files = os.listdir(folder)
    osz = ''

    # find osz
    with tempfile.TemporaryDirectory() as temp:
        filename = None
        for file in files:
            filepath = os.path.join(folder, file)

            # Find next osz
            if filepath.endswith('.osz'):
                osz = filepath
                filename = 'unknown'

                # unpack osz and find osus
                with tempfile.TemporaryDirectory() as temp2:
                    shutil.unpack_archive(osz, temp2, 'zip')
                    for f in os.listdir(temp2):
                        fpath = os.path.join(temp2, f)
                        if not fpath.endswith('.osu'): filename = '.'.join(f.split('.')[:-1])
                        if not os.path.exists(os.path.join(temp, f)): shutil.move(fpath, temp)

            os.remove(filepath)

        assert filename is not None
        shutil.make_archive(f"{filename}", 'zip', root_dir=temp)
        shutil.move(f"{filename}.zip", os.path.join(out_dir, f"{filename}.osz"))

        # elif file.endswith('.osu'):
        #     osus.append(os.path.join(folder, file))

    # for osu_file in osus:
        # with open(osu_file, 'r+') as f:
        #     osu = f.read()
        #     osu = osu.replace("Version:Mapperatorinator V30", "Version:Mapperatorinator V30\nBeatmapID:0\nBeatmapSetID:-1")


        #     # write
        #     f.seek(0)
        #     f.write(osu)
        #     f.truncate()




# ---------------------------------------------------------------------------- #
#                                      run                                     #
# ---------------------------------------------------------------------------- #
def run(file: str):
    try:
        print(f"\n----------------------------\nprocessing {file}\n")
        if not os.path.exists(file):
            raise FileNotFoundError(file)

        os.chdir(f"{MAPINATOR_DIR}/Mapperatorinator")
        with tempfile.TemporaryDirectory() as tempdir:

            for preset in presets:
                try:
                    subprocess.run(preset(file, tempdir), check=True)
                except Exception as e:
                    tb = ''.join(traceback.format_exception(e))
                    print(f"ERROR FOPR PRESET {preset}:\n{tb}")

            packgod(tempdir, f"{MAPINATOR_DIR}/out")

    except Exception as e:
        tb = ''.join(traceback.format_exception(e))
        print(f"ERROR FOPR FILE {file}:\n{tb}")

### random song suggester AGI

In [5]:
SONGS_DIR = "/run/media/jj/Ext2TB/Files/Main/Music/Tracks"
import random
os.path.join(SONGS_DIR, random.choice(os.listdir(SONGS_DIR)))

'/run/media/jj/Ext2TB/Files/Main/Music/Tracks/The Hard Way - BLCKMTL.mp3'

# Don't forget that script directory is changed!
Use absolute paths only


what I meant is that I use os.chdir so script path is changed, and therefore relative paths become wrong

In [ ]:
ESPTEIN_FILES = [
    "/run/media/jj/Ext2TB/Files/Main/Music/Tracks/The Outsiders - In Need.mp3",
    "/run/media/jj/Ext2TB/Files/Main/Music/Tracks/The Hard Way - BLCKMTL.mp3",
]


for file in ESPTEIN_FILES:
    run(file)


----------------------------
processing /run/media/jj/Ext2TB/Files/Main/Music/Tracks/The Outsiders - In Need.mp3

/run/media/jj/Ext2TB/Files/Extra/Programming2/Libs/Mapperatorinator/Mapperatorinator/.venv/bin/python inference.py -cn v32 device='cuda' audio_path='/run/media/jj/Ext2TB/Files/Main/Music/Tracks/The Outsiders - In Need.mp3' output_path='/tmp/tmp2nl13jn7' gamemode=0 difficulty=6 export_osz=true add_to_beatmap=false hitsounded=true year=2023 super_timing=false approach_rate=10 artist='The Outsiders' title='In Need' creator='Mapperatorinator (2023 6.0)'
Using SDPA for attention (auto-selected).
Random seed: 57739
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using default preview_time -1
Using gamemode-specific model checkpoint: OliBomby/Mapperatorinator-v32/gamemode=0
Mod

100%|██████████| 114/114 [00:03<00:00, 34.54it/s, 462.6 tok/s]


Precomputing encoder outputs for 114 windows...
Encoder precompute: 1.63s (14.3 ms/window)
Generating map


100%|██████████| 114/114 [00:30<00:00,  3.74it/s, 416.1 tok/s]


Generated .osz saved to /tmp/tmp2nl13jn7/beatmap863d0cd0973c46a1b7bc8eedadb54da4.osz
/run/media/jj/Ext2TB/Files/Extra/Programming2/Libs/Mapperatorinator/Mapperatorinator/.venv/bin/python inference.py -cn v32 device='cuda' audio_path='/run/media/jj/Ext2TB/Files/Main/Music/Tracks/The Outsiders - In Need.mp3' output_path='/tmp/tmp2nl13jn7' gamemode=0 difficulty=6 export_osz=true add_to_beatmap=false hitsounded=true year=2024 super_timing=false approach_rate=10 artist='The Outsiders' title='In Need' creator='Mapperatorinator (default 6.0)'
Using SDPA for attention (auto-selected).
Random seed: 31529
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using default preview_time -1
Using gamemode-specific model checkpoint: OliBomby/Mapperatorinator-v32/gamemode=0
Model loaded: OliBomby/Mappera

...

Generated .osz saved to /tmp/tmp2nl13jn7/beatmap015afe782a9c470b9a3de27bc5d29a66.osz
/run/media/jj/Ext2TB/Files/Extra/Programming2/Libs/Mapperatorinator/Mapperatorinator/.venv/bin/python inference.py -cn v32 device='cuda' audio_path='/run/media/jj/Ext2TB/Files/Main/Music/Tracks/The Outsiders - In Need.mp3' lora_path='OliBomby/Mapperatorinator-v32-LoRA-Kroytz' output_path='/tmp/tmp2nl13jn7' gamemode=0 difficulty=6 export_osz=true add_to_beatmap=false hitsounded=true descriptors=['skillset/jumps','style/clean'] year=2024 super_timing=false cfg_scale=2 approach_rate=10 artist='The Outsiders' title='In Need' creator='Mapperatorinator (LoRA-Kroytz clean jumps 6.0 cfg2)'
Using SDPA for attention (auto-selected).
Random seed: 19539
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using defau

100%|██████████| 114/114 [00:03<00:00, 30.56it/s, 373.3 tok/s]


Precomputing encoder outputs for 114 windows...
Encoder precompute: 1.61s (14.2 ms/window)
Generating map


100%|██████████| 114/114 [00:37<00:00,  3.04it/s, 338.2 tok/s]


Generated .osz saved to /tmp/tmp2nl13jn7/beatmapab8b82372d7e4b72ab8df0894d242e19.osz


In [ ]:
run("/run/media/jj/Ext2TB/Files/Main/Music/Tracks/Xtrah - Soundclash VIP.mp3")


----------------------------
processing /run/media/jj/Ext2TB/Files/Main/Music/Tracks/Xtrah - Soundclash VIP.mp3

/run/media/jj/Ext2TB/Files/Extra/Programming2/Libs/Mapperatorinator/Mapperatorinator/.venv/bin/python inference.py -cn v32 device='cuda' audio_path='/run/media/jj/Ext2TB/Files/Main/Music/Tracks/Xtrah - Soundclash VIP.mp3' output_path='/tmp/tmpzt6d0q3y' gamemode=0 difficulty=6 export_osz=true add_to_beatmap=false hitsounded=true year=2023 super_timing=true approach_rate=10 artist='Xtrah' title='Soundclash VIP' creator='Mapperatorinator (2023 6.0)'
Using SDPA for attention (auto-selected).
Random seed: 4394
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using default preview_time -1
Using gamemode-specific model checkpoint: OliBomby/Mapperatorinator-v32/gamemode=0
Model lo

100%|██████████| 20/20 [01:49<00:00,  5.50s/it, 455.3 tok/s]


Precomputing encoder outputs for 156 windows...
Encoder precompute: 2.08s (13.3 ms/window)
Generating map


100%|██████████| 156/156 [00:22<00:00,  7.05it/s, 305.8 tok/s]


Generated .osz saved to /tmp/tmpzt6d0q3y/beatmapfe7148204f924d89a6d3201a4a80f765.osz
/run/media/jj/Ext2TB/Files/Extra/Programming2/Libs/Mapperatorinator/Mapperatorinator/.venv/bin/python inference.py -cn v32 device='cuda' audio_path='/run/media/jj/Ext2TB/Files/Main/Music/Tracks/Xtrah - Soundclash VIP.mp3' output_path='/tmp/tmpzt6d0q3y' gamemode=0 difficulty=6 export_osz=true add_to_beatmap=false hitsounded=true year=2024 super_timing=true approach_rate=10 artist='Xtrah' title='Soundclash VIP' creator='Mapperatorinator (default 6.0)'
Using SDPA for attention (auto-selected).
Random seed: 25285
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using default preview_time -1
Using gamemode-specific model checkpoint: OliBomby/Mapperatorinator-v32/gamemode=0
Model loaded: OliBomby/Mapperator

 70%|███████   | 14/20 [01:17<00:33,  5.57s/it, 451.5 tok/s]

In [4]:
while True:
    path = os.path.join(SONGS_DIR, random.choice(os.listdir("/var/mnt/issd/files/music/tracks")))
    run(path)

FileNotFoundError: [Errno 2] No such file or directory: '/var/mnt/issd/files/music/tracks'